# Experimento individual e baseline

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="04_treinamento_individual"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_name': 'hi.demo.notebook.a9291c01',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': True,
 'confirmation': 'EXECUTE_DEMO',
 'request_key': 'notebooks-a9291c01',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': True,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Selecionar snapshot, algoritmo ridge/dummy e alpha. Uma campanha com um item mantém idempotência e execução independente do kernel.

In [2]:
from src.data.journey import extract
_,snapshot=extract(cfg);display(snapshot)
scopes=call(cfg,"GET","/v1/audit/scopes")
display(pd.DataFrame(scopes))
model_keys=[params.model_name]
if params.model_name=="planilha-turmas-demo":
    model_keys=[scopes[0]["model_key"]]
display({"model":params.model_name,"algorithm":params.algorithm,"alpha":params.alpha,"baseline":"DummyRegressor mean"})


{'id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
 'cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
 'target_index': 4052,
 'target_semester': 1,
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'query_sha256': 'defc10d1b6ea752600af271285c54107663e91db9df13914aea7e42613c83adf',
 'query_version': 'journey-history-v1'}

,catalog_id,campus_id,course_id,period_id,campus_code,course_code,period_number,period_code,model_family,model_key,model_display_name
0,c8454d20-98fc-5cf4-91bd-8e88b6c3c209,44bf155f-0e52-5044-bf25-e2cf4ca73019,49386655-1d9f-5160-a775-8605d1fd0d5b,ca85135b-5d7f-5b48-9607-d21bef7b19e7,DEMO-C1,DEMO-CR1,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...
1,89021437-58a6-5166-9899-086fe6f333b1,44bf155f-0e52-5044-bf25-e2cf4ca73019,49386655-1d9f-5160-a775-8605d1fd0d5b,d857c32c-2ede-57b0-8f4b-c518c6f58b0c,DEMO-C1,DEMO-CR1,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR1.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR1 • Pe...
2,871f4e77-eafd-5fb4-a094-9419851165a2,44bf155f-0e52-5044-bf25-e2cf4ca73019,6208faec-f9b8-5784-82ab-bbbb46bd8d28,c4119e4d-2581-55b4-8247-851618d67d24,DEMO-C1,DEMO-CR2,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR2.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR2 • Pe...
3,d4acd4b1-99c5-5f6d-a2a6-1ca872717462,44bf155f-0e52-5044-bf25-e2cf4ca73019,6208faec-f9b8-5784-82ab-bbbb46bd8d28,5dbb1a48-f1c2-5ede-81d4-d9ce7822f847,DEMO-C1,DEMO-CR2,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR2.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR2 • Pe...
4,b2fc44f8-5fe3-5bd1-b373-0a8af04e7d6e,44bf155f-0e52-5044-bf25-e2cf4ca73019,a5739734-976b-561d-b4c3-acbb5ed5cf83,b30529c9-d516-5198-9cee-d437f659e7ff,DEMO-C1,DEMO-CR3,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR3.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR3 • Pe...
5,b43f2bcc-74ce-5c54-91ee-8cd32eba1290,44bf155f-0e52-5044-bf25-e2cf4ca73019,a5739734-976b-561d-b4c3-acbb5ed5cf83,003a4014-8321-5d09-b9bc-0f04d766cc80,DEMO-C1,DEMO-CR3,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR3.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR3 • Pe...
6,01d4a060-7041-598e-a863-2e7a9114fe9e,44bf155f-0e52-5044-bf25-e2cf4ca73019,710ed550-edf1-5b07-baff-f2484c029fd7,1242140c-7db4-52dd-97c1-8d59049bc649,DEMO-C1,DEMO-CR4,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR4.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR4 • Pe...
7,7f3d893e-681a-5281-9a82-7fe4d7ce65c9,44bf155f-0e52-5044-bf25-e2cf4ca73019,710ed550-edf1-5b07-baff-f2484c029fd7,156d48f4-518d-56a5-979d-583889c73b00,DEMO-C1,DEMO-CR4,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR4.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR4 • Pe...
8,8080b7a3-2bd6-5acd-8899-0f9d95df3847,44bf155f-0e52-5044-bf25-e2cf4ca73019,09786e25-3a0e-5ebe-805c-207a72d98b74,de0c4144-8a18-5c60-8573-743cb74009ce,DEMO-C1,DEMO-CR5,1,P1,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR5.P1,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR5 • Pe...
9,75ad2cd2-e90b-50ba-bbbe-ecb868862ab2,44bf155f-0e52-5044-bf25-e2cf4ca73019,09786e25-3a0e-5ebe-805c-207a72d98b74,92cbdefd-c378-5e89-b349-680d8e40f22a,DEMO-C1,DEMO-CR5,2,P2,demanda-alunos,hi.pt.demanda.DEMO-C1.DEMO-CR5.P2,Planilha Turmas DEMO • DEMO-C1 • DEMO-CR5 • Pe...


{'model': 'hi.demo.notebook.a9291c01',
 'algorithm': 'ridge',
 'alpha': 1,
 'baseline': 'DummyRegressor mean'}

2. Submeter e acompanhar treino real opt-in; candidato não é promovido.

In [3]:
from src.workbench.operations_sdk import campaign
if params.execute_write:
    params.authorize()
    if not params.snapshot_id:
        raise ValueError("Selecione snapshot_id da tabela SQL")
    payload=dict(snapshot_id=str(params.snapshot_id),cycle_code=cfg.cycle,model_keys=model_keys,
        algorithm=params.algorithm,alpha=params.alpha,max_parallel_jobs=params.parallel_jobs,
        requested_by=params.requester,confirm_training=True,campaign_display_name=params.campaign_display_name)
    display({"preflight":payload,"budget":"máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas"})
    submitted=campaign(cfg,payload,params.request_key)
    campaign_id=submitted["campaign_id"]
else:
    campaign_id=str(params.campaign_id) if params.campaign_id else ""
if campaign_id:
    state=await_job(cfg,f"/v1/training/campaigns/{campaign_id}",timeout=7200)
    items=call(cfg,"GET",f"/v1/training/campaigns/{campaign_id}/items?limit=500")
    display(state);display(pd.DataFrame(items))
    evidence.update(campaign=state,items=items)
    if params.execute_write and state["status"]!="COMPLETED":
        report(NOTEBOOK_ID,evidence,status="FAIL",generation="v3")
        raise RuntimeError("Campanha não completou: "+state["status"]+"; consulte os erros dos jobs persistidos")
else:
    evidence["training"]="NOT_EXECUTED: escolha campanha existente ou habilite escrita"


{'preflight': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'cycle_code': '2026-1',
  'model_keys': ['hi.demo.notebook.a9291c01'],
  'algorithm': 'ridge',
  'alpha': 1,
  'max_parallel_jobs': 2,
  'requested_by': 'scientist-demo',
  'confirm_training': True,
  'campaign_display_name': 'POC DEMO • Validação manual'},
 'budget': 'máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas'}

{'status': 'RUNNING', 'counts': {'RUNNING': 1}, 'utc': '2026-11-04T21:43:05.691861+00:00'}


{'status': 'COMPLETED', 'counts': {'COMPLETED': 1}, 'utc': '2026-11-04T21:43:07.724782+00:00'}


{'campaign_id': '4c559b77-6699-4fe2-9ade-dce8ecd33c90',
 'idempotency_key': 'notebooks-a9291c01',
 'request_hash': 'fc1a1f250bb4ec7c406ee8e694aed3ae4864d3efef16a4a08cd0506f9aa0864d',
 'request': {'alpha': 1.0,
  'algorithm': 'ridge',
  'cycle_code': '2026-1',
  'model_keys': ['hi.demo.notebook.a9291c01'],
  'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'max_attempts': 3,
  'requested_by': 'scientist-demo',
  'schema_version': 'poc-training-campaign-v2',
  'confirm_training': True,
  'max_parallel_jobs': 2,
  'campaign_display_name': 'POC DEMO • Validação manual'},
 'correlation_id': '3f1c5325-3afc-40b5-bded-0207ebb993c7',
 'requested_by': 'scientist-demo',
 'status': 'COMPLETED',
 'max_parallel_jobs': 2,
 'requested_count': 1,
 'created_at_utc': '2026-11-04T21:43:05.656881+00:00',
 'started_at_utc': '2026-11-04T21:43:05.676607+00:00',
 'ended_at_utc': '2026-11-04T21:43:07.123537+00:00',
 'counts': {'COMPLETED': 1}}

,job_id,campaign_id,model_key,status,attempt,heartbeat_at_utc,started_at_utc,ended_at_utc,mlflow_run_id,registered_model_name,model_version,error_code,result
0,e7cdf170-3841-4794-a1b7-f454f9c7cf2a,4c559b77-6699-4fe2-9ade-dce8ecd33c90,hi.demo.notebook.a9291c01,COMPLETED,1,2026-11-04T21:43:06.562016+00:00,2026-11-04T21:43:05.676607+00:00,2026-11-04T21:43:07.106945+00:00,420b1e103c8741ddbb8ec66427a95c0b,hi.demo.notebook.a9291c01,1,None,"{'run_id': '420b1e103c8741ddbb8ec66427a95c0b',..."


3. Comparar métricas, prova de download/checksum e reload feitas pelo job. Se executado, o resultado deve ter três evidências de arquivo por modelo.

In [4]:
for item in evidence.get("items",[]):
    if item["status"]=="COMPLETED":
        display(item["result"])
        assert len(item["result"]["storage_evidence"])==3
        display(Markdown(f"[Run]({cfg.ui}/#/experiments/{item['result']['experiment_id']}/runs/{item['mlflow_run_id']})"))
display_locations(cfg,evidence.get("items",[]))


{'run_id': '420b1e103c8741ddbb8ec66427a95c0b',
 'catalog': {'domain': 'planilha-turmas',
  'run_id': '420b1e103c8741ddbb8ec66427a95c0b',
  'run_name': 'hi.demo.notebook.a9291c01__ridge__20261104T214305Z__a01__e7cdf170',
  'algorithm': 'ridge',
  'model_key': 'hi.demo.notebook.a9291c01',
  'cycle_code': '2026-1',
  'config_hash': 'f6e5e1fe48c795d059feccfe6fdb0d886c03a03c891d9c8eb137b1bcd76016ff',
  'environment': 'poc',
  'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'solution_id': 'horarios-inteligentes',
  'source_type': 'DEMO',
  'artifact_uri': 'models:/m-053c00350db44eb1a28c5dae0ed54a66',
  'code_version': 'sha256:d1a2c43715541bde0987fbdc282d0715e97950e29f8b1c6ec5137a5bcf41a628',
  'execution_id': 'e7cdf170-3841-4794-a1b7-f454f9c7cf2a',
  'manifest_uri': 's3://pucpr-poc/lineage/poc/planilha-turmas/catalog/cycles/2026-1/models/hi.demo.notebook.a9291c01/versions/v0001/manifest.json',
  'model_family': 'demanda-alunos',
  'ruleset_hash': '128eb5d2188245d1650f60b0c847efa7550

[Run](http://127.0.0.1:5000/#/experiments/3/runs/420b1e103c8741ddbb8ec66427a95c0b)

### Onde conferir

,destination,label,reference
0,Modelo DEMO,Ensaio DEMO agregado • hi.demo.notebook.a9291c01,hi.demo.notebook.a9291c01/1
1,Experimento,HI/POC/PlanilhaTurmas/Demanda/2026-1,http://127.0.0.1:5000/#/experiments/3
2,Run,hi.demo.notebook.a9291c01__ridge__20261104T214...,http://127.0.0.1:5000/#/experiments/3/runs/420...
3,Registry,hi.demo.notebook.a9291c01 • versão 1,http://127.0.0.1:5000/#/models/hi.demo.noteboo...
4,RustFS,Console • catálogo e arquivos reais,http://localhost:9001
5,Manifest,Índice imutável HI-LINEAGE/1.1,s3://pucpr-poc/lineage/poc/planilha-turmas/cat...
6,Arquivo RustFS,MLmodel,s3://pucpr-poc/mlflow/3/models/m-053c00350db44...
7,Arquivo RustFS,model.pkl,s3://pucpr-poc/mlflow/3/models/m-053c00350db44...
8,Arquivo RustFS,requirements.txt,s3://pucpr-poc/mlflow/3/models/m-053c00350db44...
9,PostgreSQL,planejamento_db.planning.v_model_inventory,SELECT * FROM planning.v_model_inventory WHERE...


[Experimento](http://127.0.0.1:5000/#/experiments/3)

[Run](http://127.0.0.1:5000/#/experiments/3/runs/420b1e103c8741ddbb8ec66427a95c0b)

[Registry](http://127.0.0.1:5000/#/models/hi.demo.notebook.a9291c01/versions/1)

[RustFS](http://localhost:9001)

[API GET Bearer](http://localhost:8000/v1/audit/model?model_key=hi.demo.notebook.a9291c01&version=1)

[Dashboard](http://localhost:8088)

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [5]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/04_treinamento_individual.json'